## Import & Config & Path settings

In [1]:
# %%
import sys
import os
import numpy as np
import pandas as pd
from pathlib import Path
import warnings

# Suppress warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# --- 1. Project Path Setup ---
def find_project_root(name='Bambino', start=None):
    if start is None: start = os.getcwd()
    parent = start
    while True:
        if os.path.basename(parent) == name: return parent
        if os.path.dirname(parent) == parent: return None
        parent = os.path.dirname(parent)

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT: 
    sys.path.append(PROJECT_ROOT)
    print(f"Project Root found: {PROJECT_ROOT}")
else:
    print("Warning: Project root 'Bambino' not found.")

# --- 2. Imports ---
import config_ad as cfg
import data_utils_ad
import moment_ad_utils
import vis_utils
from config import settings as global_settings

# %% --- 3. Load Data & Create AD Splits ---
print("\n>>> Step 1: Loading Data & Creating AD Splits")
train_loader, val_loader, test_loader = data_utils_ad.get_ad_dataloaders()
print(f"Dataset Type: {cfg.TRAIN_SPLIT_RATIO*100}% Normal for Training")

# Quick Verification of shapes
sample_x, sample_y, sample_meta = next(iter(train_loader))
print(f"\nBatch Verification:")
print(f"  X Shape: {sample_x.shape} (Expected: [B, 38, 512])")
print(f"  Y Shape: {sample_y.shape} (Should be all {cfg.NORMAL_CLASS})")

Project Root found: /home/phd2/Scrivania/CorsoRepo/Bambino

>>> Step 1: Loading Data & Creating AD Splits
Loading and Pooling all datasets...
   -> Loading: training_set.pt...
   -> Loading: validation_set.pt...
   -> Loading: test_set.pt...
Total Unique Subjects found: 46
Subject Split:
  Train Subjects: 36
  Val Subjects:   5
  Test Subjects:  5

Subject-Independent AD Data Specs:
  Train (Normal Only): 564 samples
  Val   (Normal Only): 79 samples
  Test  (Mixed):       95 samples
      -> Normals:   76
      -> Anomalies: 19
Dataset Type: 75.0% Normal for Training

Batch Verification:
  X Shape: torch.Size([16, 38, 512]) (Expected: [B, 38, 512])
  Y Shape: torch.Size([16]) (Should be all 1)


In [ ]:
sample_y

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

## Moment initialization

In [3]:
# %% --- 2. Initialize Model ---
detector = moment_ad_utils.MomentAnomalyDetector()

Loading MOMENT (Reconstruction) on cuda...


## Zero-Shot

In [12]:
# print some specs of the test loader
print(f"\nTest Loader Specs:")
print(f"  Number of Batches: {len(test_loader)}")
print(f"  Batch Size: {test_loader.batch_size}")
print(f"  Total Samples: {len(test_loader.dataset)}")
print(f"  Normal Samples: {sum(test_loader.dataset.labels == cfg.NORMAL_CLASS)}")
print(f"  Anomalous Samples: {sum(test_loader.dataset.labels != cfg.NORMAL_CLASS)}")


Test Loader Specs:
  Number of Batches: 6
  Batch Size: 16
  Total Samples: 95
  Normal Samples: 76
  Anomalous Samples: 19


In [13]:
# %% --- 3. Zero-Shot Evaluation ---
print("\n>>> Step 2: Zero-Shot Evaluation (Before Fine-tuning)")
mse_zs, lbl_zs, vis_zs = detector.predict(test_loader)
print("Zero-Shot evaluation completed.")


>>> Step 2: Zero-Shot Evaluation (Before Fine-tuning)
Running Inference...


100%|██████████| 6/6 [00:08<00:00,  1.37s/it]

Zero-Shot evaluation completed.


In [14]:
# print some specs of mse_zs and lbl_zs and vis_zs
print(f"\nZero-Shot Evaluation Specs:")
print(f"  MSE Shape: {mse_zs.shape} (Expected: [{len(test_loader.dataset)}])")
print(f"  Labels Shape: {lbl_zs.shape} (Expected: [{len(test_loader.dataset)}])")
print(f"  Visualizations Count: {len(vis_zs)} (Expected: [{len(test_loader.dataset)}])")

# print the first 5 mse_zs and lbl_zs and vis_zs
print(f"\nZero-Shot Evaluation Samples:")
print(f"  First n MSE: {mse_zs[300:310]}")
print(f"  First n Labels: {lbl_zs[300:310]}")



Zero-Shot Evaluation Specs:
  MSE Shape: (95,) (Expected: [95])
  Labels Shape: (95,) (Expected: [95])
  Visualizations Count: 3 (Expected: [95])

Zero-Shot Evaluation Samples:
  First n MSE: []
  First n Labels: []


In [15]:
# Visualize Zero-Shot
result_dir_zero_shot = os.path.join(cfg.BASE_DIR, "zero_shot_results")
os.makedirs(result_dir_zero_shot, exist_ok=True)

# 1. Global Heatmap Summary (The requested 38 channel view)
vis_utils.plot_reconstruction_heatmaps(vis_zs, result_dir_zero_shot, prefix="ZeroShot")

# 2. Error Distribution
vis_utils.plot_error_distribution(mse_zs, lbl_zs, result_dir_zero_shot, prefix="ZeroShot")

# 3. Detailed Line Plots (Optional but good)
vis_utils.plot_channel_examples(vis_zs, result_dir_zero_shot, prefix="ZeroShot")

print("Zero-Shot visualization completed.")
print(f"Results saved to {result_dir_zero_shot}")

Zero-Shot visualization completed.
Results saved to /home/phd2/Scrivania/CorsoRepo/Bambino/_03_train/moment_anomaly_detection/zero_shot_results


## Fine-Tuning

In [4]:
# %% --- 4. Fine-Tuning ---
print("\n>>> Step 3: Fine-Tuning on Normal Data (Stimulus Only)")
detector.fine_tune(train_loader, val_loader)


>>> Step 3: Fine-Tuning on Normal Data (Stimulus Only)
Starting Fine-tuning...


Epoch 1/100 [Train]: 100%|██████████| 36/36 [00:51<00:00,  1.44s/it]


Epoch 1 -> Train Loss: 0.138423 | Val Loss: 0.090976


Epoch 2/100 [Train]: 100%|██████████| 36/36 [00:51<00:00,  1.44s/it]


Epoch 2 -> Train Loss: 0.100176 | Val Loss: 0.078419


...

Epoch 33 -> Train Loss: 0.071579 | Val Loss: 0.049879


Epoch 34/100 [Train]: 100%|██████████| 36/36 [00:51<00:00,  1.43s/it]


Epoch 34 -> Train Loss: 0.071500 | Val Loss: 0.050232


Epoch 35/100 [Train]: 100%|██████████| 36/36 [00:50<00:00,  1.42s/it]


Epoch 35 -> Train Loss: 0.072517 | Val Loss: 0.050114
Early stopping triggered!
Fine-tuning Complete.


In [5]:
# %% --- 5. Fine-Tuned Evaluation ---
print("\n>>> Step 4: Fine-Tuned Evaluation")
mse_ft, lbl_ft, vis_ft = detector.predict(test_loader)
print("Fine-Tuned evaluation completed.")


>>> Step 4: Fine-Tuned Evaluation
Running Inference...


100%|██████████| 6/6 [00:08<00:00,  1.36s/it]

Fine-Tuned evaluation completed.


In [6]:
# print some specs of mse_zs and lbl_zs and vis_zs
print(f"\nFine-Tuned Evaluation Specs:")
print(f"  MSE Shape: {mse_ft.shape} (Expected: [{len(test_loader.dataset)}])")
print(f"  Labels Shape: {lbl_ft.shape} (Expected: [{len(test_loader.dataset)}])")
print(f"  Visualizations Count: {len(vis_ft)} (Expected: [{len(test_loader.dataset)}])")

# print the first 5 mse_zs and lbl_zs for class 0 and 5 for class 1
print(f"\nFine-Tuned Evaluation Samples:")
class_0_indices = np.where(lbl_ft == cfg.NORMAL_CLASS)[0][:5]
class_1_indices = np.where(lbl_ft != cfg.NORMAL_CLASS)[0][:5]
print(f"  Class {cfg.NORMAL_CLASS} MSE: {mse_ft[class_0_indices]}")
print(f"  Class {cfg.NORMAL_CLASS} Labels: {lbl_ft[class_0_indices]}")
print(f"  Class {cfg.ANOMALY_CLASS} MSE: {mse_ft[class_1_indices]}")
print(f"  Class {cfg.ANOMALY_CLASS} Labels: {lbl_ft[class_1_indices]}")


Fine-Tuned Evaluation Specs:
  MSE Shape: (95,) (Expected: [95])
  Labels Shape: (95,) (Expected: [95])
  Visualizations Count: 3 (Expected: [95])

Fine-Tuned Evaluation Samples:
  Class 1 MSE: [0.02501356 0.02911951 0.02866111 0.02538295 0.01948095]
  Class 1 Labels: [1 1 1 1 1]
  Class 0 MSE: [0.02612334 0.02331769 0.02021261 0.02049205 0.0408623 ]
  Class 0 Labels: [0 0 0 0 0]


In [7]:
# Visualize Fine-Tuned
result_dir_fine_tuned = os.path.join(cfg.BASE_DIR, "fine_tuned_results")
os.makedirs(result_dir_fine_tuned, exist_ok=True)

# 1. Global Heatmap Summary (The requested 38 channel view)
vis_utils.plot_reconstruction_heatmaps(vis_ft, result_dir_fine_tuned, prefix="FineTuned")

# 2. Error Distribution
vis_utils.plot_error_distribution(mse_ft, lbl_ft, result_dir_fine_tuned, prefix="FineTuned")

# 3. Detailed Line Plots (Optional but good)
vis_utils.plot_channel_examples(vis_ft, result_dir_fine_tuned, prefix="FineTuned")

print("Fine-Tuned visualization completed.")
print(f"Results saved to {result_dir_fine_tuned}")

Fine-Tuned visualization completed.
Results saved to /home/phd2/Scrivania/CorsoRepo/Bambino/_03_train/moment_anomaly_detection/fine_tuned_results
